In [32]:
import logging
import pandas as pd
import wbdata
from datetime import datetime

logging.getLogger("wbdata").setLevel(logging.ERROR)
##on peut definir les pays
pays_afrique = {
    "BEN": "Benin",
    "BFA": "Burkina Faso",
    "CMR": "Cameroon",
    "CIV": "Cote d'Ivoire",
    "GHA": "Ghana",
    "KEN": "Kenya",
    "NGA": "Nigeria",
    "SEN": "Senegal",
    "TGO": "Togo",
}

##difinir les indiateur a recuperer
indicators = {
    "SH.DYN.AIDS.ZS": "VIH_Prevalence",
    "SH.TBS.INCD": "Tuberculose_Incidence",
    "SH.MLR.INCD.P3": "Malaria_Incidence",
    "SH.DTH.NCOM.ZS": "Mortalite_Maladies_Chroniques",
    "SH.IMM.HEPB": "Vaccination_Hepatite_B",
    "SH.DTH.NCOM.ZS": "Mortalite_Maladies_Chroniques",
    "SP.DYN.LE00.IN": "Esperance_Vie",
    "SH.XPD.CHEX.GD.ZS": "Depenses_Sante",
    "SP.POP.TOTL": "Population"
}

# Récupération dynamique de l'année en cours
annee_actuelle = datetime.now().year
    
#Connexion à la Banque Mondiale
df = wbdata.get_dataframe(
    indicators,
    #country=list(pays_afrique.keys()),
    #date=(datetime(2000, 1, 1), datetime(annee_actuelle, 1, 1))
    date=(datetime(2000, 1, 1), datetime(2024, 1, 1))
)
#fetch des donnnes terminer
print("Export terminé")

Export terminé


In [33]:
#Nettoyage pour eviter que certains informations de se retrouver dans les index
df = df.reset_index()

#chnager le nom de la colone date
df = df.rename(columns={
    "date": "Annee",
    "country": "Pays"
})

#denition du format de l'annee
df["Annee"] = pd.to_datetime(df["Annee"]).dt.year

In [34]:
df.shape

(6650, 10)

In [35]:
df.head()

,Pays,Annee,VIH_Prevalence,Tuberculose_Incidence,Malaria_Incidence,Mortalite_Maladies_Chroniques,Vaccination_Hepatite_B,Esperance_Vie,Depenses_Sante,Population
0,Africa Eastern and Southern,2024,4.234058,NaN,196.206693,NaN,73.735115,65.349930,NaN,769280888.0
1,Africa Eastern and Southern,2023,4.415063,NaN,190.534369,NaN,73.768785,65.146228,5.540810,750491370.0
2,Africa Eastern and Southern,2022,4.594211,NaN,178.484954,NaN,74.689514,64.487152,5.559021,731821393.0
3,Africa Eastern and Southern,2021,4.750166,NaN,181.809133,36.617977,74.626897,62.979999,5.942595,713090928.0
4,Africa Eastern and Southern,2020,4.901338,NaN,181.624155,39.769347,76.915277,63.766484,5.984597,694446100.0


In [36]:
#voir le type des column
df.info()

<class 'wbdata.client.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Pays                           6650 non-null   str    
 1   Annee                          6650 non-null   int32  
 2   VIH_Prevalence                 4420 non-null   float64
 3   Tuberculose_Incidence          5632 non-null   float64
 4   Malaria_Incidence              3477 non-null   float64
 5   Mortalite_Maladies_Chroniques  1398 non-null   float64
 6   Vaccination_Hepatite_B         5408 non-null   float64
 7   Esperance_Vie                  6625 non-null   float64
 8   Depenses_Sante                 5716 non-null   float64
 9   Population                     6625 non-null   float64
dtypes: float64(8), int32(1), str(1)
memory usage: 576.7 KB


In [37]:
#Verification des valeurs null
df.isnull().sum()

Pays                                0
Annee                               0
VIH_Prevalence                   2230
Tuberculose_Incidence            1018
Malaria_Incidence                3173
Mortalite_Maladies_Chroniques    5252
Vaccination_Hepatite_B           1242
Esperance_Vie                      25
Depenses_Sante                    934
Population                         25
dtype: int64

In [38]:
df.isnull().mean() * 100

Pays                              0.000000
Annee                             0.000000
VIH_Prevalence                   33.533835
Tuberculose_Incidence            15.308271
Malaria_Incidence                47.714286
Mortalite_Maladies_Chroniques    78.977444
Vaccination_Hepatite_B           18.676692
Esperance_Vie                     0.375940
Depenses_Sante                   14.045113
Population                        0.375940
dtype: float64

In [39]:
#Mortalité maladies transmissibles (79%) on va donc supp cette colonne 
#df.drop(columns=["Mortalite_Maladies_Transmissibles"], inplace=True)
#par contre on va garder les autre valeur null ainsi afin d'avoir une analyse reel

In [40]:
#Vérification des types
df.dtypes
#aucune transformation a faire

Pays                                 str
Annee                              int32
VIH_Prevalence                   float64
Tuberculose_Incidence            float64
Malaria_Incidence                float64
Mortalite_Maladies_Chroniques    float64
Vaccination_Hepatite_B           float64
Esperance_Vie                    float64
Depenses_Sante                   float64
Population                       float64
dtype: object

In [41]:
#Détection et suppression des doublons
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [42]:
import plotly.express as px

#Évolution dans le temps 
fig = px.line(
    df,
    x="Annee",
    y="Esperance_Vie",
    color="Pays",
    title="Évolution regionnale de l'espérance"
)
fig.show()

In [43]:
#compaison entre pays
fig = px.bar(
    df[df["Annee"] == 2020],
    x="Pays",
    y="VIH_Prevalence",
    title="VIH par pays (2020)"
)
fig.show()

In [44]:
#corrélations entre tes variables

fig = px.imshow(df.corr(numeric_only=True), text_auto=True)
fig.show()

In [45]:
#nuage de piont
fig = px.scatter(
    df,
    x="Depenses_Sante",
    y="Esperance_Vie",
    color="Pays",
    title="Dépenses santé vs Espérance de vie"
)
fig.show()

In [46]:
#Top pays les plus touchés par le vih
top = df.groupby("Pays")["VIH_Prevalence"].mean().sort_values(ascending=False).head(10)

fig = px.bar(top, title="Top 10 pays VIH")
fig.show()

In [48]:
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact, interactive
import pandas as pd

# Extraction de la liste unique des pays pour le menu déroulant
liste_pays = sorted(df['Pays'].unique())

# ─────────────────────────────────────────────────────────────────────────────
# RAPPORT PAR PAYS — Visualisations 1 à 8
# ─────────────────────────────────────────────────────────────────────────────

def generer_rapport_pays(pays_choisi):
    print(f"📌 Rapport d'analyse : {pays_choisi}")
    print("-" * 50)
    
    # Filtrage du DataFrame et tri chronologique
    df_pays = df[df['Pays'] == pays_choisi].sort_values(by="Annee")
    
    # ── VIZ 1 : Évolution de l'espérance de vie (Graphique linéaire classique) ──
    fig1 = px.line(
        df_pays, x="Annee", y="Esperance_Vie",
        title=f"Évolution de l'espérance de vie - {pays_choisi}",
        markers=True, template="plotly_white"
    )
    fig1.show()

    # ── VIZ 2 : Vue d'ensemble comparative (Courbes superposées) ──
    fig2 = px.line(
        df_pays, x="Annee", 
        y=["VIH_Prevalence", "Tuberculose_Incidence", "Malaria_Incidence"],
        title=f"Comparaison des tendances épidémiologiques (Échelle globale) - {pays_choisi}",
        labels={"value": "Indice", "variable": "Pathologie"},
        markers=True
    )
    fig2.show()

    # ── VIZ 3 : Graphique à facettes — axes Y autonomes ──
    df_maladies = df_pays.melt(
        id_vars=["Annee"], 
        value_vars=["VIH_Prevalence", "Tuberculose_Incidence", "Malaria_Incidence"],
        var_name="Maladie", 
        value_name="Valeur"
    )
    fig_facettes = px.line(
        df_maladies, x="Annee", y="Valeur", 
        facet_col="Maladie", color="Maladie",
        title=f"Analyses individuelles des pathologies (Axes Y indépendants) - {pays_choisi}",
        markers=True, template="plotly_white",
        width=1100, height=450
    )
    fig_facettes.update_yaxes(matches=None, autorange=True)
    fig_facettes.show()

    # ── VIZ 4 : Focus isolé — Prévalence du VIH (Graphique en escalier) ──
    fig_vih = px.line(
        df_pays, x="Annee", y="VIH_Prevalence",
        title=f"Détail : Prévalence du VIH (% population 15-49 ans) - {pays_choisi}",
        line_shape="hv", markers=True,
        color_discrete_sequence=["#E74C3C"],
        template="plotly_white"
    )
    fig_vih.show()

    # ── VIZ 5 : Couverture vaccinale Hépatite B (Graphique à points reliés) ──
    fig3 = px.scatter(
        df_pays, x="Annee", y="Vaccination_Hepatite_B",
        title=f"Couverture vaccinale Hépatite B (% enfants 12-23 mois) - {pays_choisi}",
        labels={"Vaccination_Hepatite_B": "Taux (%)"},
        color_discrete_sequence=["#2ECC71"],
        template="plotly_white"
    )
    fig3.update_traces(mode='lines+markers')
    fig3.update_yaxes(range=[0, 100])
    fig3.show()

    # ── VIZ 6 : Mortalité due aux maladies chroniques (Diagramme en bâtons) ──
    fig4 = px.bar(
        df_pays, x="Annee", y="Mortalite_Maladies_Chroniques",
        title=f"Mortalité liée aux maladies non transmissibles (% décès) - {pays_choisi}",
        labels={"Mortalite_Maladies_Chroniques": "Part des décès (%)"},
        color_discrete_sequence=["#8E44AD"]
    )
    fig4.update_layout(xaxis_type='category')
    fig4.show()

    # ── VIZ 7 : Dépenses de santé vs Espérance de vie (Nuage de points à bulles) ──
    fig5 = px.scatter(
        df_pays, x="Depenses_Sante", y="Esperance_Vie",
        size="Population", hover_data=["Annee"],
        title=f"Analyse Investissements Santé vs Longévité - {pays_choisi}",
        labels={"Depenses_Sante": "Dépenses (% PIB)", "Esperance_Vie": "Espérance de vie (ans)"}
    )
    fig5.show()

    # ── VIZ 8 : Matrice de corrélation complète (Heatmap) ──
    colonnes_corr = [
        "VIH_Prevalence", "Tuberculose_Incidence", "Malaria_Incidence", 
        "Mortalite_Maladies_Chroniques", "Vaccination_Hepatite_B", 
        "Esperance_Vie", "Depenses_Sante", "Population"
    ]
    colonnes_presentes = [c for c in colonnes_corr if c in df_pays.columns]
    matrice_corr = df_pays[colonnes_presentes].corr()
    fig6 = px.imshow(
        matrice_corr, text_auto=".2f", 
        color_continuous_scale='RdBu_r',
        title=f"Matrice de corrélation des indicateurs - {pays_choisi}"
    )
    fig6.show()

# Widget dropdown pour le rapport par pays
interact(generer_rapport_pays, pays_choisi=widgets.Dropdown(
    options=liste_pays,
    value=liste_pays[0] if liste_pays else None,
    description='Sélection pays :',
    disabled=False,
))


# ─────────────────────────────────────────────────────────────────────────────
# VIZ 9 : CARTE CHOROPLÈTHE — Comparaison géographique interactive
# ─────────────────────────────────────────────────────────────────────────────

# Correspondance noms Banque Mondiale → codes ISO-3
iso_map = {
    "Benin": "BEN", "Burkina Faso": "BFA", "Cameroon": "CMR",
    "Cote d'Ivoire": "CIV", "Ghana": "GHA", "Kenya": "KEN",
    "Nigeria": "NGA", "Senegal": "SEN", "Togo": "TGO"
}

indicateurs_carte = {
    "Esperance_Vie":          "Espérance de vie (ans)",
    "VIH_Prevalence":         "Prévalence VIH (%)",
    "Tuberculose_Incidence":  "Incidence Tuberculose (/ 100k hab.)",
    "Malaria_Incidence":      "Incidence Malaria (/ 1k personnes à risque)",
    "Depenses_Sante":         "Dépenses de Santé (% PIB)",
    "Vaccination_Hepatite_B": "Vaccination Hépatite B (%)",
}

def afficher_carte(indicateur, annee):
    df_carte = df[df['Annee'] == annee].copy()
    df_carte['ISO'] = df_carte['Pays'].map(iso_map)
    df_carte = df_carte.dropna(subset=['ISO', indicateur])

    fig_carte = px.choropleth(
        df_carte,
        locations="ISO",
        color=indicateur,
        hover_name="Pays",
        color_continuous_scale="YlOrRd",
        scope="africa",
        title=f"🗺️ Carte : {indicateurs_carte[indicateur]} — {annee}",
        labels={indicateur: indicateurs_carte[indicateur]},
        height=580
    )
    fig_carte.update_layout(
        geo=dict(showframe=False, showcoastlines=True, bgcolor='rgba(0,0,0,0)'),
        margin=dict(t=60, b=0, l=0, r=0)
    )
    fig_carte.show()

annees_dispo = sorted(df['Annee'].unique())

interact(
    afficher_carte,
    indicateur=widgets.Dropdown(
        options=list(indicateurs_carte.keys()),
        value="Esperance_Vie",
        description="Indicateur :",
        style={'description_width': 'initial'},
    ),
    annee=widgets.SelectionSlider(
        options=annees_dispo,
        value=2019,
        description="Année :",
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
)


# ─────────────────────────────────────────────────────────────────────────────
# VIZ 10 : RADAR CHART — Profil sanitaire multi-pays
# ─────────────────────────────────────────────────────────────────────────────

# Axes positifs (plus c'est haut, mieux c'est)
axes_positifs = {
    "Esperance_Vie":          "Espérance Vie",
    "Vaccination_Hepatite_B": "Vaccination HepB",
    "Depenses_Sante":         "Dépenses Santé",
}
# Axes inversés (plus c'est bas, mieux c'est → on inverse la normalisation)
axes_inverses = {
    "VIH_Prevalence":        "VIH (inv.)",
    "Tuberculose_Incidence": "Tuberculose (inv.)",
    "Malaria_Incidence":     "Malaria (inv.)",
}

COULEURS_RADAR = ["#3498DB", "#E74C3C", "#2ECC71", "#F39C12", "#9B59B6"]

def afficher_radar(pays_selectionnes, annee):
    if not pays_selectionnes:
        print("⚠️ Veuillez sélectionner au moins un pays.")
        return

    df_annee = df[df['Annee'] == annee].copy()

    # Normalisation min-max sur les 9 pays (base commune)
    tous_axes = {**axes_positifs, **axes_inverses}
    for col in tous_axes:
        if col in df_annee.columns:
            col_min = df_annee[col].min()
            col_max = df_annee[col].max()
            if col_max != col_min:
                if col in axes_inverses:
                    df_annee[col + "_norm"] = 1 - (df_annee[col] - col_min) / (col_max - col_min)
                else:
                    df_annee[col + "_norm"] = (df_annee[col] - col_min) / (col_max - col_min)
            else:
                df_annee[col + "_norm"] = 0.5

    categories  = list(axes_positifs.values()) + list(axes_inverses.values())
    cols_norm   = [c + "_norm" for c in list(axes_positifs.keys()) + list(axes_inverses.keys())]

    fig_radar = go.Figure()

    for i, pays in enumerate(pays_selectionnes):
        row = df_annee[df_annee['Pays'] == pays]
        if row.empty:
            continue
        valeurs = [
            float(row[c].values[0]) if c in row.columns and not pd.isna(row[c].values[0]) else 0
            for c in cols_norm
        ]
        valeurs_fermees = valeurs + [valeurs[0]]

        fig_radar.add_trace(go.Scatterpolar(
            r=valeurs_fermees,
            theta=categories + [categories[0]],
            fill='toself',
            name=pays,
            line_color=COULEURS_RADAR[i % len(COULEURS_RADAR)],
            opacity=0.85
        ))

    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        showlegend=True,
        title=f"🕸️ Profil sanitaire comparé — {annee}",
        height=550,
        template="plotly_white"
    )
    fig_radar.show()
    print("ℹ️  VIH, Tuberculose et Malaria sont inversés : score élevé = faible prévalence = bonne performance.")

# Widgets radar
w_pays_radar = widgets.SelectMultiple(
    options=liste_pays,
    value=liste_pays[:3],
    description="Pays (multi) :",
    rows=9,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px', height='200px')
)
w_annee_radar = widgets.SelectionSlider(
    options=annees_dispo,
    value=2019,
    description="Année :",
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

interact(afficher_radar, pays_selectionnes=w_pays_radar, annee=w_annee_radar)


# ─────────────────────────────────────────────────────────────────────────────
# VIZ 11 : COMPARATEUR DIRECT — 2 Pays Face à Face
# ─────────────────────────────────────────────────────────────────────────────

indicateurs_comp = {
    "Esperance_Vie":          "Espérance de vie (ans)",
    "VIH_Prevalence":         "Prévalence VIH (%)",
    "Tuberculose_Incidence":  "Incidence Tuberculose",
    "Malaria_Incidence":      "Incidence Malaria",
    "Depenses_Sante":         "Dépenses Santé (% PIB)",
    "Vaccination_Hepatite_B": "Vaccination Hépatite B (%)",
}

def comparer_deux_pays(pays_A, pays_B):
    if pays_A == pays_B:
        print("⚠️ Veuillez choisir deux pays différents.")
        return

    df_A = df[df['Pays'] == pays_A].sort_values("Annee")
    df_B = df[df['Pays'] == pays_B].sort_values("Annee")

    print(f"\n⚖️  Comparaison : {pays_A}  🔵  vs  🔴  {pays_B}")
    print("=" * 60)

    # ── Grille 2 colonnes de courbes superposées ──
    indicateurs_list = list(indicateurs_comp.items())
    for i in range(0, len(indicateurs_list), 2):
        paire = indicateurs_list[i:i+2]
        figures = []

        for ind_key, ind_label in paire:
            fig_cmp = go.Figure()

            if ind_key in df_A.columns:
                fig_cmp.add_trace(go.Scatter(
                    x=df_A["Annee"], y=df_A[ind_key],
                    mode='lines+markers', name=pays_A,
                    line=dict(color="#3498DB", width=2.5),
                    marker=dict(size=5)
                ))
            if ind_key in df_B.columns:
                fig_cmp.add_trace(go.Scatter(
                    x=df_B["Annee"], y=df_B[ind_key],
                    mode='lines+markers', name=pays_B,
                    line=dict(color="#E74C3C", width=2.5, dash='dash'),
                    marker=dict(size=5)
                ))

            fig_cmp.update_layout(
                title=ind_label,
                template="plotly_white",
                height=320,
                width=560,
                margin=dict(t=45, b=30, l=40, r=20),
                legend=dict(orientation="h", yanchor="bottom", y=1.02)
            )
            figures.append(fig_cmp)

        # Afficher les graphiques séquentiellement
        for f in figures:
            f.show()

    # ── KPIs synthétiques — dernière année commune disponible ──
    derniere_annee = min(df_A['Annee'].max(), df_B['Annee'].max())
    row_A = df_A[df_A['Annee'] == derniere_annee]
    row_B = df_B[df_B['Annee'] == derniere_annee]

    print(f"\n📊 KPIs comparatifs — Dernière année disponible : {derniere_annee}")
    print(f"{'Indicateur':<35} {'🔵 ' + pays_A:>18} {'🔴 ' + pays_B:>18} {'Écart (A - B)':>15}")
    print("-" * 90)

    for ind_key, ind_label in indicateurs_comp.items():
        val_A = row_A[ind_key].values[0] if not row_A.empty and ind_key in row_A.columns and not pd.isna(row_A[ind_key].values[0]) else None
        val_B = row_B[ind_key].values[0] if not row_B.empty and ind_key in row_B.columns and not pd.isna(row_B[ind_key].values[0]) else None

        str_A   = f"{val_A:.2f}"  if val_A is not None else "N/A"
        str_B   = f"{val_B:.2f}"  if val_B is not None else "N/A"

        if val_A is not None and val_B is not None:
            delta = val_A - val_B
            signe = "▲" if delta > 0 else "▼"
            str_d = f"{signe} {abs(delta):.2f}"
        else:
            str_d = "—"

        print(f"{ind_label:<35} {str_A:>18} {str_B:>18} {str_d:>15}")

# Widgets comparateur
w_pays_A = widgets.Dropdown(
    options=liste_pays, value=liste_pays[0],
    description="🔵 Pays A :",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
w_pays_B = widgets.Dropdown(
    options=liste_pays, value=liste_pays[1],
    description="🔴 Pays B :",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

interact(comparer_deux_pays, pays_A=w_pays_A, pays_B=w_pays_B)

interactive(children=(Dropdown(description='Sélection pays :', options=('Afghanistan', 'Africa Eastern and Sou…

interactive(children=(Dropdown(description='Indicateur :', options=('Esperance_Vie', 'VIH_Prevalence', 'Tuberc…

interactive(children=(SelectMultiple(description='Pays (multi) :', index=(0, 1, 2), layout=Layout(height='200p…

interactive(children=(Dropdown(description='🔵 Pays A :', layout=Layout(width='300px'), options=('Afghanistan',…

<function __main__.comparer_deux_pays(pays_A, pays_B)>